# Tag article entities with GLiNER

Extract **company name**, **date**, **email**, and **person name** from the English and Vietnamese Markdown files in `.data`.

Run `uv sync` from the repository root, open this notebook, select the project's `.venv` kernel, and run all cells. The first model load downloads weights from Hugging Face; no API key is required. Inference runs locally on CPU.

This notebook uses [urchade/gliner_multi-v2.1](https://huggingface.co/urchade/gliner_multi-v2.1), a multilingual GLiNER model. Predictions are candidates for review, especially for Vietnamese names and company/venue distinctions. An absent email prediction is valid when the source contains no email address.

In [1]:
import csv
import html
import json
from collections import Counter
from pathlib import Path

from gliner import GLiNER

from data_platform.digest.gliner import extract_entities
from IPython.display import HTML, display

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / ".data").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the repository root or notebooks directory.")

W1117 05:14:45.877000 81617 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
/Users/chuminhquan/projects/data-platform/.venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## Select inputs and settings

By default, process all top-level `.md` files in `.data`. To select one file, set `input_paths = [data_dir / "article-en-01.md"]` (or `article-vi-01.md`). Lower `threshold` for more candidates; raise it for fewer, higher-confidence predictions. Scores are model confidence, not guaranteed probabilities of correctness.

Exports go to `.data/gliner/`; rerunning the export cell replaces those outputs. Source files stay intact. Set `local_files_only=True` after caching the model to disallow downloads.

In [2]:
data_dir = project_root / ".data"
input_paths = sorted(data_dir.glob("*.md"))
model_id = "urchade/gliner_multi-v2.1"
labels = ["company name", "date", "email", "person name"]
threshold = 0.5
chunk_words = 200
chunk_overlap = 40
local_files_only = False
output_dir = data_dir / "gliner"

if not input_paths:
    raise FileNotFoundError(f"No Markdown files found in {data_dir}")
assert 0 < threshold < 1
articles = {path.name: path.read_text(encoding="utf-8") for path in input_paths}
for name, text in articles.items():
    print(f"{name}: {len(text):,} characters")

article-en-01.md: 6,835 characters
article-vi-01.md: 4,920 characters


In [3]:
model = GLiNER.from_pretrained(model_id, local_files_only=local_files_only)
model.to("cpu")
model.eval()
print(f"Loaded {model_id} on CPU")

/Users/chuminhquan/projects/data-platform/.venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

spm.model: downloading bytes:           |  0.00B            

Loaded urchade/gliner_multi-v2.1 on CPU


## Extract entities across the full article

Chunks use GLiNER's own word splitter and overlap to retain context around boundaries. Offsets refer to the original Markdown text (zero-based Python character indices, exclusive `end`). Exact duplicate predictions keep their highest score. Conflicting overlapping spans keep the higher-scoring span so the highlighted preview is unambiguous. Chunk boundaries can still affect predictions.

In [4]:
results = {}
for name, text in articles.items():
    results[name] = extract_entities(
        text, model=model, labels=labels, threshold=threshold,
        chunk_words=chunk_words, chunk_overlap=chunk_overlap,
    )
    counts = Counter(entity["label"] for entity in results[name])
    print(name, {label: counts[label] for label in labels})

article-en-01.md {'company name': 11, 'date': 2, 'email': 0, 'person name': 4}
article-vi-01.md {'company name': 11, 'date': 0, 'email': 0, 'person name': 0}


## Review tags and confidence scores

Choose an article below. The preview preserves the raw Markdown text so its highlights match saved offsets. All displayed source text is HTML-escaped.

In [8]:
preview_name = input_paths[0].name
colors = {"company name": "#fff0ad", "date": "#cce7ff",
          "email": "#d8f5d0", "person name": "#ead7ff"}


def highlighted_html(text, entities):
    pieces = []
    cursor = 0
    for entity in entities:
        pieces.append(html.escape(text[cursor:entity["start"]]))
        label = html.escape(entity["label"])
        color = colors.get(entity["label"], "#eeeeee")
        pieces.append(
            f'<mark style="background:{color}" title="{label}: {entity["score"]:.3f}">'
            + html.escape(text[entity["start"]:entity["end"]])
            + f' <small>[{label}]</small></mark>'
        )
        cursor = entity["end"]
    pieces.append(html.escape(text[cursor:]))
    return '<div style="white-space:pre-wrap;line-height:1.9">' + ''.join(pieces) + '</div>'


entities = results[preview_name]
display(HTML(f"<h3>{html.escape(preview_name)}</h3>"))
display(HTML(highlighted_html(articles[preview_name], entities)))
rows = ''.join(
    '<tr>' + ''.join(f'<td>{html.escape(str(value))}</td>' for value in
                    (e["text"], e["label"], f'{e["score"]:.3f}', e["start"], e["end"])) + '</tr>'
    for e in entities
)
display(HTML('<table><tr><th>Text</th><th>Label</th><th>Score</th><th>Start</th><th>End</th></tr>'
             + rows + '</table>' if entities else '<p>No entities above the threshold.</p>'))

Text,Label,Score,Start,End
28-10-2026,date,0.527,109,119
"July 31, 2027",date,0.682,484,497
Mr. Nguyen Viet Quang,person name,0.682,1373,1394
Mr. Dan Levy,person name,0.699,1844,1856
Mr. Peter Loxton,person name,0.805,2290,2306
Vingroup,company name,0.695,3934,3942
Legends Global,company name,0.739,3947,3961
VinFast,company name,0.657,3963,3970
Mr. Paul Sergeant OBE,person name,0.789,4518,4539
VINGROUP,company name,0.600,4911,4919


## Save JSON, CSV, and highlighted HTML

Each JSON includes the source text, model/settings, and entities so offsets can be checked later. CSV contains one row per mention; an article with no detections has a header-only CSV. Dates retain their source spelling and are not normalized.

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
for name, entities in results.items():
    stem = Path(name).stem
    payload = {
        "source_file": name, "text": articles[name], "model": model_id,
        "labels": labels, "threshold": threshold,
        "chunk_words": chunk_words, "chunk_overlap": chunk_overlap,
        "entities": entities,
    }
    (output_dir / f"{stem}.entities.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    with (output_dir / f"{stem}.entities.csv").open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["text", "label", "score", "start", "end"])
        writer.writeheader()
        writer.writerows(entities)
    (output_dir / f"{stem}.tagged.html").write_text(
        '<!doctype html><html><head><meta charset="utf-8"><title>'
        + html.escape(name) + '</title></head><body>'
        + highlighted_html(articles[name], entities) + '</body></html>', encoding="utf-8"
    )
print(f"Saved outputs to {output_dir}")